# BF16 Colab validation
A100 80GB required. Read repository README and THIRD_PARTY_NOTICES before running. Qwen Image is research-only. Run cells in order. This notebook is generated from reviewed scripts; original evidence is separate. No Drive or token required.


In [ ]:
import threading
from google.colab import runtime
shutdown_timer=threading.Timer(25*60, runtime.unassign)
shutdown_timer.daemon=True
shutdown_timer.start()
print("25-minute best-effort runtime shutdown armed. Export results promptly.")


In [ ]:
%pip install -q transformers==5.22.1 accelerate==1.15.0 huggingface_hub==1.33.0 safetensors==0.8.0


If Colab asks to restart the session after installation, restart and rerun from the timer cell. Keep Colab supplied PyTorch. Validated environment was Python 3.13.16 / torch 2.14.1+cu130; other environments are not validated.


In [ ]:
# A100 BF16 preflight. No model weights, Drive mount, secrets or public tunnels.
import os, sys, json, time, pathlib, shutil, subprocess, platform, urllib.request
from datetime import datetime, timezone
import torch
import psutil
REPORT_DIR = pathlib.Path("/content/a100_validation")
REPORT_DIR.mkdir(exist_ok=True)
report = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0], "torch": torch.__version__, "gpu": [], "ram_gib": round(psutil.virtual_memory().total / 2**30, 2), "available_ram_gib": round(psutil.virtual_memory().available / 2**30,2), "disk_free_gib": round(shutil.disk_usage("/content").free / 2**30,2)}
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        report["gpu"].append({"name":p.name,"vram_gib":round(p.total_memory/2**30,2),"capability":list(torch.cuda.get_device_capability(i)),"bf16_supported":torch.cuda.is_bf16_supported()})
report["models"] = {}
for repo in ["huihui-ai/Huihui-Qwen3.8-27B-abliterated", "Qwen/Qwen-Image-2.1"]:
    with urllib.request.urlopen("https://huggingface.co/api/models/"+repo, timeout=30) as r:
        data=json.load(r)
    report["models"][repo]={"revision":data.get("sha"),"gated":data.get("gated"),"license":data.get("cardData",{}).get("license"),"safetensors":data.get("safetensors")}
print(json.dumps(report,ensure_ascii=False,indent=2))
(REPORT_DIR/"preflight.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
assert report["gpu"], "STOP: no GPU. No model download permitted."
assert all("A100" in g["name"] and g["bf16_supported"] for g in report["gpu"]), "STOP: A100 BF16 required."
print("PREFLIGHT_ONLY: no model has been loaded or tested.")


assert all(g["vram_gib"] > 70 for g in report["gpu"]), "STOP: A100 80GB required; 40GB is not validated."


In [ ]:
# Main generation path BF16; MTP auxiliary branch is not loaded or enabled.
# Transformers omits the optional MTP branch; audit_checkpoint.py records this limitation.
# Independent kill switch: release this paid runtime after 25 minutes if unattended.
import os, threading, time, json, gc, collections, importlib.metadata
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"]="1"
from google.colab import runtime
if "gpu_shutdown_timer" in globals():
    gpu_shutdown_timer.cancel()
gpu_shutdown_timer=threading.Timer(25*60, runtime.unassign)
gpu_shutdown_timer.daemon=True
gpu_shutdown_timer.start()
print("AUTO_SHUTDOWN: 25 minutes maximum from this cell start.")
import transformers
from transformers import AutoConfig, AutoTokenizer, AutoModelForImageTextToText
MODEL_ID="huihui-ai/Huihui-Qwen3.8-27B-abliterated"
REVISION="739e3c5b89849f6c238ce1e5b70008612ae42cdd"
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
assert torch.cuda.get_device_properties(0).total_memory/2**30 > 70
config=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,token=False)
assert not getattr(config,"quantization_config",None), "STOP: quantized checkpoint."
print("MODEL:",MODEL_ID,"REVISION:",REVISION)
print("ARCHITECTURE:",config.architectures,"MODEL_TYPE:",config.model_type)
print("VERSIONS:",{p:importlib.metadata.version(p) for p in ["torch","transformers","accelerate","huggingface_hub","safetensors"]})
started=time.monotonic()
torch.cuda.reset_peak_memory_stats()
model=AutoModelForImageTextToText.from_pretrained(MODEL_ID,revision=REVISION,dtype=torch.bfloat16,device_map={"":0},trust_remote_code=False,token=False,attn_implementation="sdpa")
model.eval()
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,token=False)
dtype_counts=collections.Counter()
device_counts=collections.Counter()
for p in model.parameters():
    dtype_counts[str(p.dtype)]+=p.numel()
    device_counts[str(p.device)]+=p.numel()
print("LOADED_PARAMETERS_BY_DTYPE:",dict(dtype_counts))
print("LOADED_PARAMETERS_BY_DEVICE:",dict(device_counts))
print("QUANTIZED:",getattr(model,"is_quantized",False))
assert not getattr(model,"is_quantized",False)
assert set(dtype_counts)=={"torch.bfloat16"}, "STOP: unexpected parameter dtype."
assert set(device_counts)=={"cuda:0"}, "STOP: not fully on A100."
results={"model":MODEL_ID,"revision":REVISION,"gpu":torch.cuda.get_device_name(0),"dtype_counts":dict(dtype_counts),"device_counts":dict(device_counts),"quantized":False,"load_seconds":round(time.monotonic()-started,2),"tests":[]}
for prompt in ["请用一句中文解释什么是 Google Colab。","计算 17 × 23，只输出整数。","Write a Python function named square that returns x multiplied by itself."]:
    encoded=tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=True,add_generation_prompt=True,enable_thinking=False,return_dict=True,return_tensors="pt").to("cuda")
    t=time.monotonic()
    with torch.inference_mode():
        output=model.generate(**encoded,max_new_tokens=128,do_sample=False,use_cache=True)
    torch.cuda.synchronize()
    generated=output[0,encoded["input_ids"].shape[1]:]
    answer=tokenizer.decode(generated,skip_special_tokens=True)
    row={"prompt":prompt,"answer":answer,"new_tokens":len(generated),"seconds":round(time.monotonic()-t,2)}
    assert answer.strip(),"STOP: empty generation."
    results["tests"].append(row)
    print("TEST:",json.dumps(row,ensure_ascii=False))
results["peak_allocated_gib"]=round(torch.cuda.max_memory_allocated()/2**30,3)
results["peak_reserved_gib"]=round(torch.cuda.max_memory_reserved()/2**30,3)
results["status"]="INFERENCE_COMPLETED_PENDING_INDEPENDENT_REVIEW"
(REPORT_DIR/"qwen3_8_bf16.json").write_text(json.dumps(results,ensure_ascii=False,indent=2))
print("QWEN_BF16_RESULT:",json.dumps(results,ensure_ascii=False,indent=2))
del model,tokenizer,encoded,output,generated,p
gc.collect()
torch.cuda.empty_cache()
print("QWEN_WEIGHTS_UNLOADED_GPU_ALLOCATED_GIB",round(torch.cuda.memory_allocated()/2**30,3))



In [ ]:
# Read-only checkpoint/header audit. No tensors loaded, no GPU inference.
import json, math, collections, pathlib, gc
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForImageTextToText
audit_repo="huihui-ai/Huihui-Qwen3.8-27B-abliterated"
audit_revision="739e3c5b89849f6c238ce1e5b70008612ae42cdd"
index_path=pathlib.Path(hf_hub_download(audit_repo,"model.safetensors.index.json",revision=audit_revision,local_files_only=True,token=False))
index=json.loads(index_path.read_text())
checkpoint={}
checkpoint_dtype=collections.Counter()
for shard in sorted(set(index["weight_map"].values())):
    with safe_open(str(index_path.parent/shard),framework="pt",device="cpu") as f:
        for key in f.keys():
            sl=f.get_slice(key)
            shape=list(sl.get_shape())
            dtype=sl.get_dtype()
            count=math.prod(shape)
            checkpoint[key]={"shape":shape,"numel":count,"dtype":dtype}
            checkpoint_dtype[dtype]+=count
audit_config=AutoConfig.from_pretrained(audit_repo,revision=audit_revision,local_files_only=True,trust_remote_code=False,token=False)
with init_empty_weights():
    meta_model=AutoModelForImageTextToText.from_config(audit_config,trust_remote_code=False,dtype=torch.bfloat16,attn_implementation="sdpa")
meta_state={k:{"shape":list(v.shape),"numel":v.numel()} for k,v in meta_model.state_dict().items()}
checkpoint_only={k:checkpoint[k] for k in checkpoint.keys()-meta_state.keys()}
model_only={k:meta_state[k] for k in meta_state.keys()-checkpoint.keys()}
shape_mismatch={k:{"checkpoint":checkpoint[k],"model":meta_state[k]} for k in checkpoint.keys()&meta_state.keys() if checkpoint[k]["shape"]!=meta_state[k]["shape"]}
checkpoint_audit={"repo":audit_repo,"revision":audit_revision,"checkpoint_dtype_numel":dict(checkpoint_dtype),"checkpoint_total":sum(v["numel"] for v in checkpoint.values()),"meta_state_dict_total":sum(v["numel"] for v in meta_state.values()),"meta_parameters_total":sum(p.numel() for p in meta_model.parameters()),"meta_parameters_without_deduplication":sum(p.numel() for n,p in meta_model.named_parameters(remove_duplicate=False)),"checkpoint_only_total":sum(v["numel"] for v in checkpoint_only.values()),"checkpoint_only":checkpoint_only,"model_only":model_only,"shape_mismatch":shape_mismatch,"note":"Headers and meta tensors only. No real tensors loaded or inference repeated."}
(REPORT_DIR/"qwen_checkpoint_audit.json").write_text(json.dumps(checkpoint_audit,indent=2))
print("QWEN_CHECKPOINT_AUDIT:",json.dumps(checkpoint_audit,indent=2))
del meta_model,sl
gc.collect()
print("POST_AUDIT_GPU_ALLOCATED_GIB",round(torch.cuda.memory_allocated()/2**30,3))



Download the ZIP before shutdown. If download fails, use Colab Files to download JSON/PNG manually. Do not leave paid GPU running while troubleshooting export. A PNG filename/hash in JSON does not prove the file was exported.


In [ ]:
# Export only this test's public-safe evidence. No accounts, secrets or weights.
import importlib.metadata, hashlib, json, shutil, pathlib
from google.colab import files
versions={}
for package in ["torch","transformers","accelerate","huggingface_hub","safetensors","diffusers","numpy","Pillow"]:
    try: versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: pass
versions["diffusers_source_commit"]="84532a058702c74ae5d1dd6ceb05683fc5d5e768"
(REPORT_DIR/"runtime_versions.json").write_text(json.dumps(versions,indent=2))
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in REPORT_DIR.iterdir() if p.is_file() and p.name != "sha256_manifest.json"}
(REPORT_DIR/"sha256_manifest.json").write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive("/content/a100_validation_evidence","zip",str(REPORT_DIR))
print("EVIDENCE_EXPORT_FILES",sorted(manifest))
print("EXPORT_SHA256",hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest())
print("Next: disconnect and delete this paid runtime; then verify zero active sessions.")
files.download(archive)



In [ ]:
from google.colab import runtime
runtime.unassign()
